# Latihan SVM (Not OK): SVM Kernel Linear

Contoh klasifikasi dua kelas: siswa **lulus** (+1) atau **tidak lulus** (-1) ujian, berdasarkan lama belajar (x1) dan lama tidur (x2).

Notebook ini berisi (1) hitungan manual dengan kode, lalu (2) implementasi dengan SKLearn, dan (3) perbandingan hasilnya.

## 1. Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm

# Dataset points
x1 = np.array([2, 1, 3, 1, 4, 2, 5, 2])  # Study Hours
x2 = np.array([8, 6, 7, 4, 6, 5, 7, 3])  # Sleep Hours
y = np.array([1, -1, 1, -1, 1, -1, 1, -1])  # Passed Exam

X = np.column_stack((x1, x2))

## 2. Perhitungan manual

Support vector yang dipilih:

| Titik | Kelas | Persamaan |
|---|---|---|
| (3, 7) | +1 | 3w1 + 7w2 + b = 1 |
| (4, 6) | +1 | 4w1 + 6w2 + b = 1 |
| (1, 6) | -1 | 1w1 + 6w2 + b = -1 |
| (2, 5) | -1 | 2w1 + 5w2 + b = -1 |

Dari selisih persamaan sekelas didapat `w1 = w2`. Lalu `10w1 + b = 1` dan `7w1 + b = -1` memberi `w1 = 2/3`, `w2 = 2/3`, `b = -17/3`.

Kodenya, menyelesaikan 4 persamaan di atas sekaligus:

In [ ]:
# 4 persamaan, 3 variabel (w1, w2, b). Diselesaikan dengan least squares.
A = np.array([[3, 7, 1],
              [4, 6, 1],
              [1, 6, 1],
              [2, 5, 1]], dtype=float)
rhs = np.array([1, 1, -1, -1], dtype=float)

sol, residual, rank, sv = np.linalg.lstsq(A, rhs, rcond=None)
w1, w2, b_manual = sol
print('w1 =', w1, '(2/3 =', 2/3, ')')
print('w2 =', w2)
print('b  =', b_manual, '(-17/3 =', -17/3, ')')
print('selisih terbesar terhadap persamaan:', np.abs(A @ sol - rhs).max())

### Garis batas dan garis margin

Garis batas: `2x1 + 2x2 - 17 = 0`, yaitu `x2 = -x1 + 8.5`.

Garis margin adalah tempat `w1x1 + w2x2 + b = +1` dan `= -1`:

- Margin positif: `(2/3)(x1 + x2) - 17/3 = 1`, jadi `x1 + x2 = 10`, yaitu **`x2 = -x1 + 10`**
- Margin negatif: `(2/3)(x1 + x2) - 17/3 = -1`, jadi `x1 + x2 = 7`, yaitu **`x2 = -x1 + 7`**

> Catatan: di materi tertulis margin `x2 = -x1 + 9` dan `x2 = -x1 + 8` (dari 8.5 ± 0.5). Itu tidak cocok dengan hitungannya sendiri. Support vector (3,7) dan (4,6) berada di `x1 + x2 = 10` dan (1,6), (2,5) berada di `x1 + x2 = 7`, jadi margin yang benar adalah +10 dan +7. Ini kemungkinan penyebab label "Not OK".

In [ ]:
x1_range = np.linspace(0, 6, 100)
decision_boundary = -x1_range + 8.5
margin_positive = -x1_range + 10
margin_negative = -x1_range + 7

plt.figure(figsize=(8, 6))
plt.scatter(x1[y == 1], x2[y == 1], color='blue', label='Passed (y=1)', s=100)
plt.scatter(x1[y == -1], x2[y == -1], color='red', label='Failed (y=-1)', s=100)

plt.plot(x1_range, decision_boundary, color='green', label='Decision Boundary', linewidth=2)
plt.plot(x1_range, margin_positive, color='orange', linestyle='--', label='Positive Margin', linewidth=1.5)
plt.plot(x1_range, margin_negative, color='purple', linestyle='--', label='Negative Margin', linewidth=1.5)

plt.title('SVM Decision Boundary and Margins (manual)', fontsize=14)
plt.xlabel('Study Hours (x1)', fontsize=12)
plt.ylabel('Sleep Hours (x2)', fontsize=12)
plt.axhline(0, color='black', linewidth=0.5, linestyle='--')
plt.axvline(0, color='black', linewidth=0.5, linestyle='--')
plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)
plt.legend(fontsize=12)
plt.xlim(0, 6)
plt.ylim(0, 10)
plt.show()

## 3. Implementasi dengan SKLearn

In [ ]:
# Create and fit the SVM model
clf = svm.SVC(kernel='linear')  # Linear kernel
clf.fit(X, y)

# Get the decision boundary (hyperplane)
w = clf.coef_[0]
b = clf.intercept_[0]

print('w =', w)
print('b =', b)
print('support vectors:')
print(clf.support_vectors_)

In [ ]:
# Plot dengan hasil SKLearn: batas (nilai 0) dan margin (nilai +1 dan -1)
x1_range = np.linspace(0, 6, 100)
decision_boundary = -(w[0] * x1_range + b) / w[1]
margin_pos = -(w[0] * x1_range + b - 1) / w[1]
margin_neg = -(w[0] * x1_range + b + 1) / w[1]

plt.figure(figsize=(8, 6))
plt.scatter(x1[y == 1], x2[y == 1], color='blue', label='Passed (y=1)', s=100)
plt.scatter(x1[y == -1], x2[y == -1], color='red', label='Failed (y=-1)', s=100)
plt.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
            s=250, facecolors='none', edgecolors='black', label='Support Vectors')

plt.plot(x1_range, decision_boundary, color='green', label='Decision Boundary', linewidth=2)
plt.plot(x1_range, margin_pos, color='orange', linestyle='--', label='Positive Margin', linewidth=1.5)
plt.plot(x1_range, margin_neg, color='purple', linestyle='--', label='Negative Margin', linewidth=1.5)

plt.title('SVM Decision Boundary (Scikit-learn)', fontsize=14)
plt.xlabel('Study Hours (x1)', fontsize=12)
plt.ylabel('Sleep Hours (x2)', fontsize=12)
plt.axhline(0, color='black', linewidth=0.5, linestyle='--')
plt.axvline(0, color='black', linewidth=0.5, linestyle='--')
plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)
plt.legend(fontsize=11)
plt.xlim(0, 6)
plt.ylim(0, 10)
plt.show()

## 4. Perbandingan manual vs SKLearn

In [ ]:
print('Manual : w1 = %.4f, w2 = %.4f, b = %.4f' % (w1, w2, b_manual))
print('SKLearn: w1 = %.4f, w2 = %.4f, b = %.4f' % (w[0], w[1], b))
print('Akurasi training SKLearn:', clf.score(X, y))

Hasil SKLearn sama dengan hitungan manual: `w = (2/3, 2/3)` dan `b = -17/3`, sehingga garis batasnya `x2 = -x1 + 8.5`. Margin yang benar adalah `x2 = -x1 + 10` dan `x2 = -x1 + 7`.